In [16]:
import numpy as np
import pandas as pd

In [17]:
# Import specific classes from scikit-learn for data preprocessing:
# SimpleImputer for handling missing values.
# OneHotEncoder for converting categorical features into a one-hot numerical array.
# OrdinalEncoder for converting categorical features into ordinal integers.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [18]:
df = pd.read_csv('/content/covid_toy.csv')

In [19]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
13,64,Male,102.0,Mild,Bangalore,Yes
12,25,Female,99.0,Strong,Kolkata,No
65,69,Female,102.0,Mild,Bangalore,No
87,47,Male,101.0,Strong,Bangalore,No
5,84,Female,NaN,Mild,Bangalore,Yes


In [20]:
# Check for missing values in each column of the DataFrame and sum them up.
# This helps identify columns with incomplete data.
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [21]:
# Import 'train_test_split' from scikit-learn to divide the dataset into training and testing sets.
# X (features) will be all columns except 'has_covid', and y (target) will be 'has_covid'.
# test_size=0.2 means 20% of the data will be used for testing, and 80% for training.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns = ['has_covid']), df['has_covid'], test_size = 0.2)

# 1. Aam Zindagi

In [22]:
# Initialize SimpleImputer for handling missing values, which by default uses the mean strategy.
si = SimpleImputer()
# Apply the imputer to the 'fever' column of the training data (X_train) to fill missing values.
X_train_fever = si.fit_transform(X_train[['fever']])

# Apply the imputer to the 'fever' column of the test data (X_test).
X_test_fever = si.fit_transform(X_test[['fever']])

# Display the shape of the transformed 'fever' training data to verify its dimensions.
X_train_fever.shape

(80, 1)

In [23]:
# Initialize OrdinalEncoder for the 'cough' column, defining the order of categories as 'Mild' then 'Strong'.
oe = OrdinalEncoder(categories=[['Mild','Strong']])
# Apply the encoder to the 'cough' column of the training data.
X_train_cough = oe.fit_transform(X_train[['cough']])

# Apply the encoder to the 'cough' column of the test data.
X_test_cough = oe.fit_transform(X_test[['cough']])

# Display the shape of the transformed 'cough' training data.
X_train_cough.shape

(80, 1)

In [24]:
# Initialize OneHotEncoder for 'gender' and 'city' columns.
# drop='first' prevents multicollinearity by dropping the first category.
# sparse_output=False ensures a dense NumPy array output.
ohe = OneHotEncoder(drop='first',sparse_output=False)
# Apply the encoder to 'gender' and 'city' columns of the training data.
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# Apply the encoder to 'gender' and 'city' columns of the test data.
X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])

# Display the shape of the transformed 'gender' and 'city' training data.
X_train_gender_city.shape

(80, 4)

In [25]:
# Extract the 'age' column from the training data by dropping all other processed columns.
# .values converts the DataFrame column into a NumPy array.
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# Extract the 'age' column from the test data.
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

# Display the shape of the extracted 'age' training data.
X_train_age.shape

(80, 1)

In [26]:
# Concatenate all individually transformed feature arrays for the training data along axis=1 (columns).
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)

# Concatenate all individually transformed feature arrays for the test data.
X_test_transformed = np.concatenate((X_test_age, X_test_fever, X_test_gender_city, X_test_cough), axis = 1)

# Display the shape of the final transformed training data.
X_train_transformed.shape

(80, 7)

# Mentos Zindagi

In [27]:
# Import ColumnTransformer from scikit-learn, which allows applying different transformers to different columns of the data.
from sklearn.compose import ColumnTransformer

In [28]:
# Define a ColumnTransformer to apply various preprocessing steps in a single pipeline.
# 'tnf1': SimpleImputer to the 'fever' column.
# 'tnf2': OrdinalEncoder to the 'cough' column with specified categories.
# 'tnf3': OneHotEncoder to 'gender' and 'city' columns, dropping the first category and returning a dense array.
# 'remainder='passthrough'': ensures that columns not explicitly transformed (like 'age') are kept in the output.
transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(),['fever']),
     ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
      ('tnf3', OneHotEncoder(sparse_output=False, drop='first'),['gender','city'])], remainder='passthrough')

In [29]:
# Apply the defined ColumnTransformer to the training data (X_train) and display the shape of the resulting transformed array.
transformer.fit_transform(X_train).shape

(80, 7)

In [30]:
# Apply the defined ColumnTransformer to the test data (X_test) and display the shape of the resulting transformed array.
transformer.transform(X_test).shape

(20, 7)